In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '4'

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from typing import List, Optional, Union, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

/home/admin01/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# 中期预测

## HOUSTON

### XGBoost

In [50]:
def short_term_forecast_XGB(horizon_number=1):
    # 模型配置（当前最佳）
    model = XGBRegressor(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.8,
        colsample_bylevel=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        min_child_weight=10,
        gamma=0,
        objective='reg:absoluteerror',
        tree_method='gpu_hist',
        enable_categorical=True,
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )

    X = pd.read_csv(f'./rtm_dam_fea_med/train_data_houston_horizon{horizon_number}.csv')
    X = X[X.date >= '2022-01-01 00:00:00']
    print(X.shape)
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X = X.dropna()
    X_train = X[X.date < '2025-01-01 00:00:00']
    X_test = X[X.date >= '2025-01-01 00:00:00']
    _ = X_train.pop('date')
    dt = X_test.pop('date')
    _ = X_train.pop('target')
    _ = X_test.pop('target')
    y_train = X_train.pop('actual_future_price')
    y_test = X_test.pop('actual_future_price')

    # cat_features = [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 48, 51, 54, 65, 66]

    # for i in cat_features:
    #     X_train.iloc[:, i] = X_train.iloc[:, i].astype("category")
    #     X_test.iloc[:, i] = X_test.iloc[:, i].astype("category")

    model.fit(
        X_train, y_train
        # categorical_feature=cat_features
    )
    
    return  model, X_test, y_test, dt

#### 未来1小时

In [51]:
model1, X_test, y_test, dt = short_term_forecast_XGB(horizon_number=1)

y_pred = model1.predict(X_test)
xgb_model_mae1 = mean_absolute_error(y_test, y_pred)
xgb_model_rmse1 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae1 = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse1 = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("xgb_model_mae1:", xgb_model_mae1)
print("native_mae:", native_mae1)

print("xgb_model_rmse1:", xgb_model_rmse1)
print("native_rmse:", native_rmse1)

xgb_mae_enhance1 = round((native_mae1 - xgb_model_mae1) / native_mae1 * 100, 2)
xgb_rmse_enhance1 = round((native_rmse1 - xgb_model_rmse1) / native_rmse1 * 100, 2)
print('mae 提升:', xgb_mae_enhance1, '%')
print('rmse 提升:', xgb_rmse_enhance1, '%')

(137940, 84)
xgb_model_mae1: 7.164919758700943
native_mae: 9.425053721014931
xgb_model_rmse1: 33.54499498011313
native_rmse: 45.56823988094555
mae 提升: 23.98 %
rmse 提升: 26.39 %


#### 预测未来2小时

In [52]:
model2, X_test, y_test, dt = short_term_forecast_XGB(horizon_number=2)

y_pred = model2.predict(X_test)
xgb_model_mae2 = mean_absolute_error(y_test, y_pred)
xgb_model_rmse2 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae2 = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse2 = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("xgb_model_mae2:", xgb_model_mae2)
print("native_mae:", native_mae2)

print("xgb_model_rmse2:", xgb_model_rmse2)
print("native_rmse:", native_rmse2)

xgb_mae_enhance2 = round((native_mae2 - xgb_model_mae2) / native_mae2 * 100, 2)
xgb_rmse_enhance2 = round((native_rmse2 - xgb_model_rmse2) / native_rmse2 * 100, 2)
print('mae 提升:', xgb_mae_enhance2, '%')
print('rmse 提升:', xgb_rmse_enhance2, '%')

(137936, 84)
xgb_model_mae2: 8.79306158138051
native_mae: 13.759043528411851
xgb_model_rmse2: 37.32513698608329
native_rmse: 51.32498321025448
mae 提升: 36.09 %
rmse 提升: 27.28 %


#### 预测未来4小时

In [53]:
model4, X_test, y_test, dt = short_term_forecast_XGB(horizon_number=4)

y_pred = model4.predict(X_test)
xgb_model_mae4 = mean_absolute_error(y_test, y_pred)
xgb_model_rmse4 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae4 = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse4 = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("xgb_model_mae4:", xgb_model_mae4)
print("native_mae:", native_mae4)

print("xgb_model_rmse4:", xgb_model_rmse4)
print("native_rmse:", native_rmse4)

xgb_mae_enhance4 = round((native_mae4 - xgb_model_mae4) / native_mae4 * 100, 2)
xgb_rmse_enhance4 = round((native_rmse4 - xgb_model_rmse4) / native_rmse4 * 100, 2)
print('mae 提升:', xgb_mae_enhance4, '%')
print('rmse 提升:', xgb_rmse_enhance4, '%')

(137928, 84)
xgb_model_mae4: 10.051777592936507
native_mae: 18.578318557201847
xgb_model_rmse4: 38.40131146156873
native_rmse: 55.307716805339
mae 提升: 45.9 %
rmse 提升: 30.57 %


#### 预测未来6小时

In [54]:
model6, X_test, y_test, dt = short_term_forecast_XGB(horizon_number=6)

y_pred = model6.predict(X_test)
xgb_model_mae6 = mean_absolute_error(y_test, y_pred)
xgb_model_rmse6 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae6 = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse6 = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("xgb_model_mae6:", xgb_model_mae6)
print("native_mae:", native_mae6)

print("xgb_model_rmse6:", xgb_model_rmse6)
print("native_rmse:", native_rmse6)

xgb_mae_enhance6 = round((native_mae6 - xgb_model_mae6) / native_mae6 * 100, 2)
xgb_rmse_enhance6 = round((native_rmse6 - xgb_model_rmse6) / native_rmse6 * 100, 2)
print('mae 提升:', xgb_mae_enhance6, '%')
print('rmse 提升:', xgb_rmse_enhance6, '%')

(137920, 84)
xgb_model_mae6: 9.97510207366086
native_mae: 20.684112305636543
xgb_model_rmse6: 38.234069746553665
native_rmse: 57.12026068730592
mae 提升: 51.77 %
rmse 提升: 33.06 %


#### 未来12小时

In [55]:
model12, X_test, y_test, dt = short_term_forecast_XGB(horizon_number=12)

y_pred = model12.predict(X_test)
xgb_model_mae12 = mean_absolute_error(y_test, y_pred)
xgb_model_rmse12 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae12 = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse12 = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("xgb_model_mae1:", xgb_model_mae12)
print("native_mae:", native_mae12)

print("xgb_model_rmse1:", xgb_model_rmse12)
print("native_rmse:", native_rmse12)

xgb_mae_enhance12 = round((native_mae12 - xgb_model_mae12) / native_mae12 * 100, 2)
xgb_rmse_enhance12 = round((native_rmse12 - xgb_model_rmse12) / native_rmse12 * 100, 2)
print('mae 提升:', xgb_mae_enhance12, '%')
print('rmse 提升:', xgb_rmse_enhance12, '%')

(137896, 84)
xgb_model_mae1: 10.338344875251524
native_mae: 22.11385545830294
xgb_model_rmse1: 38.8727791024335
native_rmse: 58.134123295203906
mae 提升: 53.25 %
rmse 提升: 33.13 %


#### 未来24小时

In [56]:
model24, X_test, y_test, dt = short_term_forecast_XGB(horizon_number=24)

y_pred = model24.predict(X_test)
xgb_model_mae24 = mean_absolute_error(y_test, y_pred)
xgb_model_rmse24 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae24 = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse24 = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("xgb_model_mae24:", xgb_model_mae24)
print("native_mae:", native_mae24)

print("xgb_model_rmse24:", xgb_model_rmse24)
print("native_rmse:", native_rmse24)

xgb_mae_enhance24 = round((native_mae24 - xgb_model_mae24) / native_mae24 * 100, 2)
xgb_rmse_enhance24 = round((native_rmse24 - xgb_model_rmse24) / native_rmse24 * 100, 2)
print('mae 提升:', xgb_mae_enhance24, '%')
print('rmse 提升:', xgb_rmse_enhance24, '%')

(137848, 84)
xgb_model_mae24: 10.10270170215159
native_mae: 16.18098368638909
xgb_model_rmse24: 38.41956545897272
native_rmse: 53.608190682016264
mae 提升: 37.56 %
rmse 提升: 28.33 %


### LightGBM

In [73]:
def short_term_forecast_LGBM(horizon_number=1):
    # 模型配置（当前最佳）
    model = LGBMRegressor(
        n_estimators=2000,
        num_leaves=31,
        max_depth=8,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.8,
        subsample_freq=1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        min_child_samples=100,
        min_child_weight=1e-3,
        objective='mae',
        metric='mae',
        device='gpu',
        boosting_type='gbdt',
        random_state=42,
        n_jobs=4,
        verbose=-1,
        force_row_wise=True,
    )

    X = pd.read_csv(f'./rtm_dam_fea_med/train_data_houston_horizon{horizon_number}.csv')
    X = X[X.date >= '2022-01-01 00:00:00']
    print(X.shape)
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X = X.dropna()
    X_train = X[X.date < '2025-01-01 00:00:00']
    X_test = X[X.date >= '2025-01-01 00:00:00']
    _ = X_train.pop('date')
    dt = X_test.pop('date')
    _ = X_train.pop('target')
    _ = X_test.pop('target')
    y_train = X_train.pop('actual_future_price')
    y_test = X_test.pop('actual_future_price')

    # cat_features = [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 48, 51, 54, 65, 66]

    # for i in cat_features:
    #     X_train.iloc[:, i] = X_train.iloc[:, i].astype("category")
    #     X_test.iloc[:, i] = X_test.iloc[:, i].astype("category")

    model.fit(
        X_train, y_train
        # categorical_feature=cat_features
    )
    
    return  model, X_test, y_test, dt

#### 未来1小时

In [74]:
model1, X_test, y_test, dt = short_term_forecast_LGBM(horizon_number=1)

y_pred = model1.predict(X_test)
lgb_model_mae1 = mean_absolute_error(y_test, y_pred)
lgb_model_rmse1 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("lgb_model_mae1:", lgb_model_mae1)
print("native_mae:", native_mae)

print("lgb_model_rmse1:", lgb_model_rmse1)
print("native_rmse:", native_rmse)

lgb_mae_enhance1 = round((native_mae - lgb_model_mae1) / native_mae * 100, 2)
lgb_rmse_enhance1 = round((native_rmse - lgb_model_rmse1) / native_rmse * 100, 2)
print('mae 提升:', lgb_mae_enhance1, '%')
print('rmse 提升:', lgb_rmse_enhance1, '%')

(137940, 84)
lgb_model_mae1: 7.080511935464782
native_mae: 9.425053721014931
lgb_model_rmse1: 32.92830684137276
native_rmse: 45.56823988094555
mae 提升: 24.88 %
rmse 提升: 27.74 %


#### 未来2小时

In [75]:
model2, X_test, y_test, dt = short_term_forecast_LGBM(horizon_number=2)

y_pred = model2.predict(X_test)
lgb_model_mae2 = mean_absolute_error(y_test, y_pred)
lgb_model_rmse2 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("lgb_model_mae1:", lgb_model_mae2)
print("native_mae:", native_mae)

print("lgb_model_rmse1:", lgb_model_rmse2)
print("native_rmse:", native_rmse)

lgb_mae_enhance2 = round((native_mae - lgb_model_mae2) / native_mae * 100, 2)
lgb_rmse_enhance2 = round((native_rmse - lgb_model_rmse2) / native_rmse * 100, 2)
print('mae 提升:', lgb_mae_enhance2, '%')
print('rmse 提升:', lgb_rmse_enhance2, '%')

(137936, 84)
lgb_model_mae1: 8.623748627833121
native_mae: 13.759043528411851
lgb_model_rmse1: 36.248737229214996
native_rmse: 51.32498321025448
mae 提升: 37.32 %
rmse 提升: 29.37 %


#### 未来4小时

In [26]:
model4, X_test, y_test, dt = short_term_forecast_LGBM(horizon_number=4)

y_pred = model4.predict(X_test)
lgb_model_mae4 = mean_absolute_error(y_test, y_pred)
lgb_model_rmse4 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("lgb_model_mae1:", lgb_model_mae4)
print("native_mae:", native_mae)

print("lgb_model_rmse1:", lgb_model_rmse4)
print("native_rmse:", native_rmse)

lgb_mae_enhance4 = round((native_mae - lgb_model_mae4) / native_mae * 100, 2)
lgb_rmse_enhance4 = round((native_rmse - lgb_model_rmse4) / native_rmse * 100, 2)
print('mae 提升:', lgb_mae_enhance4, '%')
print('rmse 提升:', lgb_rmse_enhance4, '%')

(137928, 84)
lgb_model_mae1: 9.609549322230457
native_mae: 18.578318557201847
lgb_model_rmse1: 37.95538509430774
native_rmse: 55.307716805339
mae 提升: 48.28 %
rmse 提升: 31.37 %


#### 未来6小时

In [27]:
model6, X_test, y_test, dt = short_term_forecast_LGBM(horizon_number=6)

y_pred = model6.predict(X_test)
lgb_model_mae6 = mean_absolute_error(y_test, y_pred)
lgb_model_rmse6 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("lgb_model_mae1:", lgb_model_mae6)
print("native_mae:", native_mae)

print("lgb_model_rmse1:", lgb_model_rmse6)
print("native_rmse:", native_rmse)

lgb_mae_enhance6 = round((native_mae - lgb_model_mae6) / native_mae * 100, 2)
lgb_rmse_enhance6 = round((native_rmse - lgb_model_rmse6) / native_rmse * 100, 2)
print('mae 提升:', lgb_mae_enhance6, '%')
print('rmse 提升:', lgb_rmse_enhance6, '%')

(137920, 84)
lgb_model_mae1: 10.084690106685477
native_mae: 20.684112305636543
lgb_model_rmse1: 39.870264932619506
native_rmse: 57.12026068730592
mae 提升: 51.24 %
rmse 提升: 30.2 %


#### 未来12小时

In [28]:
model12, X_test, y_test, dt = short_term_forecast_LGBM(horizon_number=12)

y_pred = model12.predict(X_test)
lgb_model_mae12 = mean_absolute_error(y_test, y_pred)
lgb_model_rmse12 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("lgb_model_mae1:", lgb_model_mae12)
print("native_mae:", native_mae)

print("lgb_model_rmse1:", lgb_model_rmse12)
print("native_rmse:", native_rmse)

lgb_mae_enhance12 = round((native_mae - lgb_model_mae12) / native_mae * 100, 2)
lgb_rmse_enhance12 = round((native_rmse - lgb_model_rmse12) / native_rmse * 100, 2)
print('mae 提升:', lgb_mae_enhance12, '%')
print('rmse 提升:', lgb_rmse_enhance12, '%')

(137896, 84)
lgb_model_mae1: 10.162541190721548
native_mae: 22.11385545830294
lgb_model_rmse1: 38.468232367495304
native_rmse: 58.134123295203906
mae 提升: 54.04 %
rmse 提升: 33.83 %


#### 未来24小时

In [29]:
model24, X_test, y_test, dt = short_term_forecast_LGBM(horizon_number=24)

y_pred = model24.predict(X_test)
lgb_model_mae24 = mean_absolute_error(y_test, y_pred)
lgb_model_rmse24 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("lgb_model_mae1:", lgb_model_mae24)
print("native_mae:", native_mae)

print("lgb_model_rmse1:", lgb_model_rmse24)
print("native_rmse:", native_rmse)

lgb_mae_enhance24 = round((native_mae - lgb_model_mae24) / native_mae * 100, 2)
lgb_rmse_enhance24 = round((native_rmse - lgb_model_rmse24) / native_rmse * 100, 2)
print('mae 提升:', lgb_mae_enhance24, '%')
print('rmse 提升:', lgb_rmse_enhance24, '%')

(137848, 84)
lgb_model_mae1: 10.01429065361836
native_mae: 16.18098368638909
lgb_model_rmse1: 38.495473150510904
native_rmse: 53.608190682016264
mae 提升: 38.11 %
rmse 提升: 28.19 %


### CatBoost

In [65]:
def short_term_forecast_CAT(horizon_number=1, cat_features=[]):
    # 模型配置（当前最佳）
    model = CatBoostRegressor(
        iterations=2000,
        depth=6,
        learning_rate=0.1,
        l2_leaf_reg=3.0,
        random_strength=1.0,
        min_data_in_leaf=20,
        subsample=0.8,
        bootstrap_type='Bernoulli',
        loss_function='MAE',
        task_type='GPU',
        random_seed=42,
        thread_count=-1,
        verbose=100,
        allow_writing_files=False,
        cat_features=cat_features,
    )
    

    X = pd.read_csv(f'./rtm_dam_fea_med/train_data_houston_horizon{horizon_number}.csv')
    X = X[X.date >= '2022-01-01 00:00:00']
    print(X.shape)
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X = X.dropna()
    X_train = X[X.date < '2025-01-01 00:00:00']
    X_test = X[X.date >= '2025-01-01 00:00:00']
    _ = X_train.pop('date')
    dt = X_test.pop('date')
    _ = X_train.pop('target')
    _ = X_test.pop('target')
    y_train = X_train.pop('actual_future_price')
    y_test = X_test.pop('actual_future_price')

    model.fit(
        X_train, y_train
    )
    
    return  model, X_test, y_test, dt

#### 预测未来1小时

In [66]:
cat_features = [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 48, 51, 54, 65, 66]
model1, X_test, y_test, dt = short_term_forecast_CAT(horizon_number=1, cat_features=cat_features)

y_pred = model1.predict(X_test)
cat_model_mae1 = mean_absolute_error(y_test, y_pred)
cat_model_rmse1 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("cat_model_mae1:", cat_model_mae1)
print("native_mae:", native_mae)

print("cat_model_rmse1:", cat_model_rmse1)
print("native_rmse:", native_rmse)

cat_mae_enhance1 = round((native_mae - cat_model_mae1) / native_mae * 100, 2)
cat_rmse_enhance1 = round((native_rmse - cat_model_rmse1) / native_rmse * 100, 2)
print('mae 提升:', cat_mae_enhance1, '%')
print('rmse 提升:', cat_rmse_enhance1, '%')

(137940, 84)


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 32.6408401	total: 343ms	remaining: 11m 25s
100:	learn: 29.6414838	total: 30.5s	remaining: 9m 34s
200:	learn: 27.6200966	total: 1m	remaining: 9m 4s
300:	learn: 26.1958694	total: 1m 30s	remaining: 8m 29s
400:	learn: 25.1258844	total: 1m 58s	remaining: 7m 51s
500:	learn: 24.2648676	total: 2m 28s	remaining: 7m 24s
600:	learn: 23.5547837	total: 2m 59s	remaining: 6m 56s
700:	learn: 22.9743840	total: 3m 25s	remaining: 6m 20s
800:	learn: 22.4782124	total: 3m 50s	remaining: 5m 44s
900:	learn: 22.0452082	total: 4m 14s	remaining: 5m 10s
1000:	learn: 21.6758887	total: 4m 36s	remaining: 4m 36s
1100:	learn: 21.3724184	total: 4m 57s	remaining: 4m 2s
1200:	learn: 21.1424897	total: 5m 19s	remaining: 3m 32s
1300:	learn: 20.9764722	total: 5m 31s	remaining: 2m 57s
1400:	learn: 20.8499228	total: 5m 55s	remaining: 2m 31s
1500:	learn: 20.7541144	total: 6m 17s	remaining: 2m 5s
1600:	learn: 20.6780269	total: 6m 38s	remaining: 1m 39s
1700:	learn: 20.6113053	total: 7m 2s	remaining: 1m 14s
1800:	learn: 

#### 预测未来2小时

In [44]:
cat_features = [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 48, 51, 54, 65, 66]
model2, X_test, y_test, dt = short_term_forecast_CAT(horizon_number=2, cat_features=cat_features)

y_pred = model2.predict(X_test)
cat_model_mae2 = mean_absolute_error(y_test, y_pred)
cat_model_rmse2 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("cat_model_mae1:", cat_model_mae2)
print("native_mae:", native_mae)

print("cat_model_rmse1:", cat_model_rmse2)
print("native_rmse:", native_rmse)

cat_mae_enhance2 = round((native_mae - cat_model_mae2) / native_mae * 100, 2)
cat_rmse_enhance2 = round((native_rmse - cat_model_rmse2) / native_rmse * 100, 2)
print('mae 提升:', cat_mae_enhance2, '%')
print('rmse 提升:', cat_rmse_enhance2, '%')

(137936, 84)


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 32.6442585	total: 12.7ms	remaining: 25.3s
100:	learn: 29.9202463	total: 1.17s	remaining: 22s
200:	learn: 28.0960491	total: 2.25s	remaining: 20.2s
300:	learn: 26.8114082	total: 3.29s	remaining: 18.6s
400:	learn: 25.8412435	total: 4.34s	remaining: 17.3s
500:	learn: 25.0614559	total: 5.66s	remaining: 16.9s
600:	learn: 24.4188089	total: 7.91s	remaining: 18.4s
700:	learn: 23.8931071	total: 10.2s	remaining: 18.9s
800:	learn: 23.4457478	total: 12.5s	remaining: 18.7s
900:	learn: 23.0570793	total: 14.8s	remaining: 18s
1000:	learn: 22.7222625	total: 17.1s	remaining: 17s
1100:	learn: 22.4466108	total: 19.3s	remaining: 15.8s
1200:	learn: 22.2383458	total: 21.5s	remaining: 14.3s
1300:	learn: 22.0844807	total: 23.8s	remaining: 12.8s
1400:	learn: 21.9663555	total: 26s	remaining: 11.1s
1500:	learn: 21.8761371	total: 28.3s	remaining: 9.41s
1600:	learn: 21.8018970	total: 30.5s	remaining: 7.61s
1700:	learn: 21.7369465	total: 32.9s	remaining: 5.78s
1800:	learn: 21.6805537	total: 35.2s	remaining:

#### 预测未来4小时

In [45]:
cat_features = [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 48, 51, 54, 65, 66]
model4, X_test, y_test, dt = short_term_forecast_CAT(horizon_number=4, cat_features=cat_features)

y_pred = model4.predict(X_test)
cat_model_mae4 = mean_absolute_error(y_test, y_pred)
cat_model_rmse4 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("cat_model_mae1:", cat_model_mae4)
print("native_mae:", native_mae)

print("cat_model_rmse1:", cat_model_rmse4)
print("native_rmse:", native_rmse)

cat_mae_enhance4 = round((native_mae - cat_model_mae4) / native_mae * 100, 2)
cat_rmse_enhance4 = round((native_rmse - cat_model_rmse4) / native_rmse * 100, 2)
print('mae 提升:', cat_mae_enhance4, '%')
print('rmse 提升:', cat_rmse_enhance4, '%')

(137928, 84)


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 32.6672332	total: 12.4ms	remaining: 24.7s
100:	learn: 30.1289594	total: 1.11s	remaining: 20.9s
200:	learn: 28.4471686	total: 2.19s	remaining: 19.6s
300:	learn: 27.2646602	total: 3.28s	remaining: 18.5s
400:	learn: 26.3670501	total: 4.47s	remaining: 17.8s
500:	learn: 25.6397174	total: 6.84s	remaining: 20.5s
600:	learn: 25.0403763	total: 9.31s	remaining: 21.7s
700:	learn: 24.5507242	total: 11.8s	remaining: 21.8s
800:	learn: 24.1400559	total: 14.2s	remaining: 21.2s
900:	learn: 23.7849580	total: 16.6s	remaining: 20.3s
1000:	learn: 23.4807106	total: 19s	remaining: 19s
1100:	learn: 23.2296307	total: 21.4s	remaining: 17.4s
1200:	learn: 23.0355062	total: 23.6s	remaining: 15.7s
1300:	learn: 22.8899748	total: 25.9s	remaining: 13.9s
1400:	learn: 22.7759568	total: 28.2s	remaining: 12.1s
1500:	learn: 22.6857384	total: 30.6s	remaining: 10.2s
1600:	learn: 22.6126211	total: 32.9s	remaining: 8.21s
1700:	learn: 22.5501163	total: 35.3s	remaining: 6.2s
1800:	learn: 22.4908916	total: 37.6s	remaini

#### 预测未来6小时

In [46]:
cat_features = [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 48, 51, 54, 65, 66]
model6, X_test, y_test, dt = short_term_forecast_CAT(horizon_number=6, cat_features=cat_features)

y_pred = model6.predict(X_test)
cat_model_mae6 = mean_absolute_error(y_test, y_pred)
cat_model_rmse6 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("cat_model_mae1:", cat_model_mae6)
print("native_mae:", native_mae)

print("cat_model_rmse1:", cat_model_rmse6)
print("native_rmse:", native_rmse)

cat_mae_enhance6 = round((native_mae - cat_model_mae6) / native_mae * 100, 2)
cat_rmse_enhance6 = round((native_rmse - cat_model_rmse6) / native_rmse * 100, 2)
print('mae 提升:', cat_mae_enhance6, '%')
print('rmse 提升:', cat_rmse_enhance6, '%')

(137920, 84)


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 32.6785037	total: 14.7ms	remaining: 29.4s
100:	learn: 30.1948468	total: 1.31s	remaining: 24.6s
200:	learn: 28.5697705	total: 2.58s	remaining: 23.1s
300:	learn: 27.4282748	total: 3.86s	remaining: 21.8s
400:	learn: 26.5493631	total: 5.13s	remaining: 20.5s
500:	learn: 25.8351625	total: 6.41s	remaining: 19.2s
600:	learn: 25.2450084	total: 7.6s	remaining: 17.7s
700:	learn: 24.7663526	total: 8.79s	remaining: 16.3s
800:	learn: 24.3635531	total: 10s	remaining: 15s
900:	learn: 24.0149820	total: 11.3s	remaining: 13.7s
1000:	learn: 23.7152447	total: 12.5s	remaining: 12.5s
1100:	learn: 23.4681839	total: 13.7s	remaining: 11.2s
1200:	learn: 23.2801261	total: 14.7s	remaining: 9.81s
1300:	learn: 23.1397365	total: 15.8s	remaining: 8.51s
1400:	learn: 23.0282691	total: 16.9s	remaining: 7.24s
1500:	learn: 22.9397121	total: 18s	remaining: 6s
1600:	learn: 22.8642993	total: 19.1s	remaining: 4.77s
1700:	learn: 22.7978470	total: 20.2s	remaining: 3.55s
1800:	learn: 22.7382456	total: 21.3s	remaining: 2

#### 未来12小时

In [47]:
cat_features = [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 48, 51, 54, 65, 66]
model12, X_test, y_test, dt = short_term_forecast_CAT(horizon_number=12, cat_features=cat_features)

y_pred = model12.predict(X_test)
cat_model_mae12 = mean_absolute_error(y_test, y_pred)
cat_model_rmse12 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("cat_model_mae1:", cat_model_mae12)
print("native_mae:", native_mae)

print("cat_model_rmse1:", cat_model_rmse12)
print("native_rmse:", native_rmse)

cat_mae_enhance12 = round((native_mae - cat_model_mae12) / native_mae * 100, 2)
cat_rmse_enhance12 = round((native_rmse - cat_model_rmse12) / native_rmse * 100, 2)
print('mae 提升:', cat_mae_enhance12, '%')
print('rmse 提升:', cat_rmse_enhance12, '%')

(137896, 84)


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 32.6697505	total: 13.3ms	remaining: 26.5s
100:	learn: 30.2188608	total: 1.21s	remaining: 22.8s
200:	learn: 28.6269499	total: 2.34s	remaining: 20.9s
300:	learn: 27.5061334	total: 3.35s	remaining: 18.9s
400:	learn: 26.6409283	total: 4.38s	remaining: 17.4s
500:	learn: 25.9369398	total: 5.39s	remaining: 16.1s
600:	learn: 25.3572958	total: 6.42s	remaining: 14.9s
700:	learn: 24.8826471	total: 7.45s	remaining: 13.8s
800:	learn: 24.4814972	total: 8.49s	remaining: 12.7s
900:	learn: 24.1371811	total: 9.53s	remaining: 11.6s
1000:	learn: 23.8442494	total: 10.6s	remaining: 10.5s
1100:	learn: 23.6011433	total: 11.6s	remaining: 9.48s
1200:	learn: 23.4193071	total: 12.6s	remaining: 8.41s
1300:	learn: 23.2734563	total: 13.7s	remaining: 7.35s
1400:	learn: 23.1573668	total: 14.7s	remaining: 6.3s
1500:	learn: 23.0691436	total: 15.7s	remaining: 5.23s
1600:	learn: 22.9915853	total: 16.7s	remaining: 4.17s
1700:	learn: 22.9148971	total: 17.7s	remaining: 3.11s
1800:	learn: 22.8518036	total: 18.7s	rem

#### 未来24小时

In [48]:
cat_features = [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 48, 51, 54, 65, 66]
model24, X_test, y_test, dt = short_term_forecast_CAT(horizon_number=24, cat_features=cat_features)

y_pred = model24.predict(X_test)
cat_model_mae24 = mean_absolute_error(y_test, y_pred)
cat_model_rmse24 = np.sqrt(mean_squared_error(y_test, y_pred))
native_mae = mean_absolute_error(y_test, X_test.iloc[:, -1])
native_rmse = np.sqrt(mean_squared_error(y_test, X_test.iloc[:, -1]))
print("cat_model_mae1:", cat_model_mae24)
print("native_mae:", native_mae)

print("cat_model_rmse1:", cat_model_rmse24)
print("native_rmse:", native_rmse)

cat_mae_enhance24 = round((native_mae - cat_model_mae24) / native_mae * 100, 2)
cat_rmse_enhance24 = round((native_rmse - cat_model_rmse24) / native_rmse * 100, 2)
print('mae 提升:', cat_mae_enhance24, '%')
print('rmse 提升:', cat_rmse_enhance24, '%')

(137848, 84)


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 32.6729781	total: 10.5ms	remaining: 21s
100:	learn: 30.2325699	total: 895ms	remaining: 16.8s
200:	learn: 28.6477531	total: 1.72s	remaining: 15.4s
300:	learn: 27.5367339	total: 2.56s	remaining: 14.4s
400:	learn: 26.6807539	total: 3.4s	remaining: 13.5s
500:	learn: 25.9821480	total: 4.25s	remaining: 12.7s
600:	learn: 25.4026326	total: 5.2s	remaining: 12.1s
700:	learn: 24.9231759	total: 6.18s	remaining: 11.4s
800:	learn: 24.5130225	total: 7.18s	remaining: 10.7s
900:	learn: 24.1587136	total: 8.18s	remaining: 9.98s
1000:	learn: 23.8489120	total: 9.18s	remaining: 9.16s
1100:	learn: 23.5910528	total: 10.2s	remaining: 8.32s
1200:	learn: 23.3914671	total: 11.2s	remaining: 7.45s
1300:	learn: 23.2373923	total: 12.2s	remaining: 6.56s
1400:	learn: 23.1164876	total: 13.2s	remaining: 5.66s
1500:	learn: 23.0195088	total: 14.3s	remaining: 4.74s
1600:	learn: 22.9367777	total: 15.3s	remaining: 3.81s
1700:	learn: 22.8636270	total: 16.3s	remaining: 2.87s
1800:	learn: 22.7918470	total: 17.3s	remain

In [76]:
native_mae = [native_mae1, native_mae2, native_mae4, native_mae6, native_mae12, native_mae24]
native_rmse = [native_rmse1, native_rmse2, native_rmse4, native_rmse6, native_rmse12, native_rmse24]


xgb_mae = [xgb_model_mae1, xgb_model_mae2, xgb_model_mae4, xgb_model_mae6, xgb_model_mae12, xgb_model_mae24]
xgb_mae_enhance = [xgb_mae_enhance1, xgb_mae_enhance2, xgb_mae_enhance4, xgb_mae_enhance6, xgb_mae_enhance12, xgb_mae_enhance24]
xgb_rmse = [xgb_model_rmse1, xgb_model_rmse2, xgb_model_rmse4, xgb_model_rmse6, xgb_model_rmse12, xgb_model_rmse24]
xgb_rmse_enhance = [xgb_rmse_enhance1, xgb_rmse_enhance2, xgb_rmse_enhance4, xgb_rmse_enhance6, xgb_rmse_enhance12, xgb_rmse_enhance24]


lgb_mae = [lgb_model_mae1, lgb_model_mae2, lgb_model_mae4, lgb_model_mae6, lgb_model_mae12, lgb_model_mae24]
lgb_mae_enhance = [lgb_mae_enhance1, lgb_mae_enhance2, lgb_mae_enhance4, lgb_mae_enhance6, lgb_mae_enhance12, lgb_mae_enhance24]
lgb_rmse = [lgb_model_rmse1, lgb_model_rmse2, lgb_model_rmse4, lgb_model_rmse6, lgb_model_rmse12, lgb_model_rmse24]
lgb_rmse_enhance = [lgb_rmse_enhance1, lgb_rmse_enhance2, lgb_rmse_enhance4, lgb_rmse_enhance6, lgb_rmse_enhance12, lgb_rmse_enhance24]


cat_mae = [cat_model_mae1, cat_model_mae2, cat_model_mae4, cat_model_mae6, cat_model_mae12, cat_model_mae24]
cat_mae_enhance = [cat_mae_enhance1, cat_mae_enhance2, cat_mae_enhance4, cat_mae_enhance6, cat_mae_enhance12, cat_mae_enhance24]
cat_rmse = [cat_model_rmse1, cat_model_rmse2, cat_model_rmse4, cat_model_rmse6, cat_model_rmse12, cat_model_rmse24]
cat_rmse_enhance = [cat_rmse_enhance1, cat_rmse_enhance2, cat_rmse_enhance4, cat_rmse_enhance6, cat_rmse_enhance12, cat_rmse_enhance24]

In [77]:
rtm_short_term_forecast_metrics = [native_mae, native_rmse,
                                   xgb_mae, xgb_mae_enhance, xgb_rmse, xgb_rmse_enhance,
                                   lgb_mae, lgb_mae_enhance, lgb_rmse, lgb_rmse_enhance,
                                   cat_mae, cat_mae_enhance, cat_rmse, cat_rmse_enhance]
rtm_short_term_forecast_metrics = pd.DataFrame(rtm_short_term_forecast_metrics).T

In [78]:
rtm_short_term_forecast_metrics.columns = ['native_mae', 'native_rmse',
                                           'xgb_mae', 'xgb_mae_enhance', 'xgb_rmse', 'xgb_rmse_enhance',
                                           'lgb_mae', 'lgb_mae_enhance', 'lgb_rmse', 'lgb_rmse_enhance',
                                           'cat_mae', 'cat_mae_enhance', 'cat_rmse', 'cat_rmse_enhance']

In [79]:
rtm_short_term_forecast_metrics.index = ['未来1小时', '未来2小时', '未来4小时', '未来6小时', '未来12小时', '未来24小时']
rtm_short_term_forecast_metrics.to_csv('metrics_result/rtm_mid_term_forecast_metrics.csv', index=None)
rtm_short_term_forecast_metrics

,native_mae,native_rmse,xgb_mae,xgb_mae_enhance,xgb_rmse,xgb_rmse_enhance,lgb_mae,lgb_mae_enhance,lgb_rmse,lgb_rmse_enhance,cat_mae,cat_mae_enhance,cat_rmse,cat_rmse_enhance
未来1小时,9.425054,45.568240,7.164920,23.98,33.544995,26.39,7.080512,24.88,32.928307,27.74,7.451217,20.94,35.846387,21.33
未来2小时,13.759044,51.324983,8.793062,36.09,37.325137,27.28,8.623749,37.32,36.248737,29.37,8.842152,35.74,37.052351,27.81
未来4小时,18.578319,55.307717,10.051778,45.90,38.401311,30.57,9.609549,48.28,37.955385,31.37,9.773372,47.39,38.027662,31.24
未来6小时,20.684112,57.120261,9.975102,51.77,38.234070,33.06,10.084690,51.24,39.870265,30.20,10.126568,51.04,38.378342,32.81
未来12小时,22.113855,58.134123,10.338345,53.25,38.872779,33.13,10.162541,54.04,38.468232,33.83,10.319528,53.33,38.634145,33.54
未来24小时,16.180984,53.608191,10.102702,37.56,38.419565,28.33,10.014291,38.11,38.495473,28.19,10.114656,37.49,38.425099,28.32
